In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/mydata/doc2validate")

# From this directory, you can also get article IDs which take part in the experiment 
GEN_ROOT = (
    ROOT
    / "results/runs/scidata_4293/generated_code"
)

rows_file = []
rows_article = []

for article_dir in sorted(GEN_ROOT.iterdir()):
    if not article_dir.is_dir():
        continue

    article_id = article_dir.name

    run_output = article_dir / "run_output.json"
    
    # rows_article is a array of  json lines
    if not run_output.exists():
        rows_article.append(
            {
                "article_id": article_id,
                "article_status": "no_run_output",
                "n_runs": 0,
                "n_load_success": 0,
                "n_validation_success": 0,
            }
        )
        continue

    data = json.loads(
        run_output.read_text(
            encoding="utf-8"
        )
    )

    runs = data.get("runs", [])

    load_success = 0
    validation_success = 0

    for r in runs:
        # logical_name ->file_info->run->runs
        file_info = r.get("file", {})
        load_result = r.get("load_result", {})
        validation = r.get("validation", {})
        profile = r.get("profile", {})

        load_ok = load_result.get("ok", False)
        validation_ok = validation.get("ok", False)

        if load_ok:
            load_success += 1

        if validation_ok:
            validation_success += 1

        rows_file.append(
            {
                "article_id": article_id,
                "logical_name": file_info.get(
                    "logical_name"
                ),
                "format": file_info.get(
                    "format"
                ),
                "schema_type": file_info.get(
                    "schema_type"
                ),
                "load_ok": load_ok,
                "validation_ok": validation_ok,
                "strategy": load_result.get(
                    "strategy"
                ),
                "selected_path": load_result.get(
                    "path"
                ),
                "load_error": load_result.get(
                    "error"
                ),
                "n_rows": profile.get(
                    "n_rows"
                ),
                "n_cols": profile.get(
                    "n_cols"
                ),
            }
        )

    if validation_success > 0:
        status = "partial_or_full_success"
    elif load_success > 0:
        status = "load_only"
    else:
        status = "failed"

    rows_article.append(
        {
            "article_id": article_id,
            "article_status": status,
            "n_runs": len(runs),
            "n_load_success": load_success,
            "n_validation_success": validation_success,
        }
    )

file_df = pd.DataFrame(rows_file)
article_df = pd.DataFrame(rows_article)

print(file_df.shape)
print(article_df.shape)

file_df.head()

(85, 11)
(32, 5)


,article_id,logical_name,format,schema_type,load_ok,validation_ok,strategy,selected_path,load_error,n_rows,n_cols
0,s41597-019-0035-4,anatomical_iqms,csv,tabular,False,False,not_found,None,file_not_found,NaN,NaN
1,s41597-019-0035-4,functional_iqms,csv,tabular,False,False,not_found,None,file_not_found,NaN,NaN
2,s41597-019-0035-4,curated_anatomical_iqms,csv,tabular,False,False,not_found,None,file_not_found,NaN,NaN
3,s41597-020-00609-9,primary_interventions_data,csv,tabular,False,False,basename_search,/mydata/doc2validate/data/downloaded_artifacts...,UnicodeDecodeError: 'utf-8' codec can't decode...,NaN,NaN
4,s41597-020-00609-9,static_dataset_snapshot,csv,tabular,True,True,extension_fallback,/mydata/doc2validate/data/downloaded_artifacts...,None,4933.0,87.0


In [2]:
OUT = ROOT / "results/runs/scidata_4293/analysis"

OUT.mkdir(exist_ok=True, parents=True)

file_df.to_csv(
    OUT / "execution_file_level.csv",
    index=False,
)

article_df.to_csv(
    OUT / "execution_article_level.csv",
    index=False,
)

print("saved")

saved


In [3]:
article_df.article_status.value_counts()

partial_or_full_success    25
failed                      6
no_run_output               1
Name: article_status, dtype: int64

In [4]:
file_df.validation_ok.value_counts()

True     63
False    22
Name: validation_ok, dtype: int64

In [5]:
(
    file_df[
        ~file_df["validation_ok"]
    ]
    .groupby(["format", "strategy"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

format  strategy          
csv     not_found             8
        basename_search       5
json    basename_search       4
tsv     not_found             3
xlsx    extension_fallback    1
        not_found             1
dtype: int64

In [6]:
failed_df = file_df[
    ~file_df.validation_ok
]

failed_df[
    [
        "article_id",
        "logical_name",
        "format",
        "strategy",
        "load_error",
        "selected_path",
    ]
].head(50)

,article_id,logical_name,format,strategy,load_error,selected_path
0,s41597-019-0035-4,anatomical_iqms,csv,not_found,file_not_found,None
1,s41597-019-0035-4,functional_iqms,csv,not_found,file_not_found,None
2,s41597-019-0035-4,curated_anatomical_iqms,csv,not_found,file_not_found,None
3,s41597-020-00609-9,primary_interventions_data,csv,basename_search,UnicodeDecodeError: 'utf-8' codec can't decode...,/mydata/doc2validate/data/downloaded_artifacts...
5,s41597-020-00609-9,master_list_of_codes,csv,basename_search,UnicodeDecodeError: 'utf-8' codec can't decode...,/mydata/doc2validate/data/downloaded_artifacts...
16,s41597-020-0455-1,study_metadata_template,xlsx,extension_fallback,None,/mydata/doc2validate/data/downloaded_artifacts...
24,s41597-022-01350-1,sentences,tsv,not_found,file_not_found,None
25,s41597-022-01350-1,entity_annotations,tsv,not_found,file_not_found,None
26,s41597-022-01350-1,relation_annotations,tsv,not_found,file_not_found,None
27,s41597-022-01432-0,wordlist_data,json,basename_search,ValueError: All arrays must be of the same length,/mydata/doc2validate/data/downloaded_artifacts...


In [7]:
def classify_failure(row):
    err = str(row.load_error)

    if "UnicodeDecodeError" in err:
        return "encoding_failure"

    if "file_not_found" in err:
        return "artifact_not_found"

    if "same length" in err:
        return "parser_structure_failure"

    if "Mixing dicts" in err:
        return "parser_structure_failure"

    if row.article_id == "s41597-019-0035-4":
        return "placeholder_repository"

    return "other"


failed_df["failure_category"] = (
    failed_df.apply(classify_failure, axis=1)
)

failed_df.failure_category.value_counts()

/tmp/ipykernel_158271/495507145.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  failed_df["failure_category"] = (


artifact_not_found          12
encoding_failure             5
parser_structure_failure     4
other                        1
Name: failure_category, dtype: int64

In [8]:
failed_df[
    failed_df.failure_category == "other"
][
    [
        "article_id",
        "logical_name",
        "load_error",
        "selected_path",
    ]
]

,article_id,logical_name,load_error,selected_path
16,s41597-020-0455-1,study_metadata_template,None,/mydata/doc2validate/data/downloaded_artifacts...


In [9]:
file_df[
    file_df.article_id == "s41597-020-0455-1"
][
    [
        "logical_name",
        "load_ok",
        "validation_ok",
        "load_error",
        "n_rows",
        "n_cols",
    ]
]

,logical_name,load_ok,validation_ok,load_error,n_rows,n_cols
14,routes_and_species_summary,True,True,None,779.0,10.0
15,unextracted_sources_list,True,True,None,779.0,10.0
16,study_metadata_template,True,False,None,0.0,15.0


In [10]:
true_failures = file_df[
    ~file_df.load_ok
].copy()

true_failures["failure_category"] = (
    true_failures.apply(
        classify_failure,
        axis=1
    )
)

true_failures.failure_category.value_counts()

artifact_not_found          12
encoding_failure             5
parser_structure_failure     4
Name: failure_category, dtype: int64

In [12]:
true_failures = file_df[
    ~file_df.load_ok
].copy()

true_failures["failure_category"] = (
    true_failures.apply(
        classify_failure,
        axis=1
    )
)

true_failures.failure_category.value_counts()

artifact_not_found          12
encoding_failure             5
parser_structure_failure     4
Name: failure_category, dtype: int64

In [13]:
import pandas as pd

df = pd.read_csv(
    "../results/runs/scidata_4293/analysis/grounding_failure_taxonomy.csv"
)

def collapse(row):
    cat = row["failure_category"]
    reason = str(row["reason"]).lower()

    if cat == "inventory_absent":
        return "inventory_absent"

    if cat == "unsupported_modality":
        return "unsupported_modality"

    if "multiple" in reason:
        return "semantic_ambiguity"

    if (
        "directory" in reason
        or "split across" in reason
        or "same directory" in reason
        or "plural" in reason
        or "version" in reason
    ):
        return "schema_granularity_mismatch"

    if (
        "not found" in reason
        or "not present" in reason
        or "no file named" in reason
        or "no file matching" in reason
    ):
        return "repository_missing"

    return "repository_missing"

df["paper_category"] = df.apply(
    collapse,
    axis=1,
)

print(
    df["paper_category"]
    .value_counts()
)

print(
    "\nPercent:"
)

print(
    (
        df["paper_category"]
        .value_counts(normalize=True)
        * 100
    )
    .round(1)
)

repository_missing             82
unsupported_modality           14
schema_granularity_mismatch     9
inventory_absent                5
semantic_ambiguity              3
Name: paper_category, dtype: int64

Percent:
repository_missing             72.6
unsupported_modality           12.4
schema_granularity_mismatch     8.0
inventory_absent                4.4
semantic_ambiguity              2.7
Name: paper_category, dtype: float64
